# MiniLM Transformer Improvement Runs

This notebook runs the MiniLM-only improvement plan for resume-job compatibility prediction.
It keeps `all-MiniLM-L6-v2` fixed, increases the training budget, adds a learning-rate scheduler through the shared module, and compares text-only, lexical-only, structured-only, and hybrid variants.

In [ ]:
%pip install -q -r transformer/requirements.txt

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'transformer').exists() and (REPO_ROOT / '4120-initial-investigation').exists():
    REPO_ROOT = REPO_ROOT / '4120-initial-investigation'

sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display

from transformer.transformer import (
    PRIMARY_SEED,
    load_dataframe,
    prepare_splits,
    run_experiment,
    summarize_token_lengths,
)

sns.set_theme(style='whitegrid')
DATA_PATH = REPO_ROOT / 'resume_data_cleaned.csv'
ARTIFACT_DIR = REPO_ROOT / 'transformer' / 'artifacts'
PLOT_DIR = ARTIFACT_DIR / 'plots'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename: str) -> None:
    fig.savefig(PLOT_DIR / filename, dpi=200, bbox_inches='tight')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'repo root: {REPO_ROOT}')
print(f'device: {device}')
print(f'data path: {DATA_PATH}')
print(f'plot dir: {PLOT_DIR}')

In [ ]:
ENCODERS = [
    'sentence-transformers/all-MiniLM-L6-v2',
    # 'distilbert-base-uncased',
]
VARIANTS = [
    {'name': 'text-only', 'use_structured': False, 'use_lexical': False},
    {'name': 'lexical-only', 'use_structured': False, 'use_lexical': True},
    {'name': 'structured-only', 'use_structured': True, 'use_lexical': False},
    {'name': 'hybrid', 'use_structured': True, 'use_lexical': True},
]

MAX_LENGTH = 256
RESUME_WORD_LIMIT = 220
JOB_WORD_LIMIT = 96
BATCH_SIZE = 16
EPOCHS = 10
PATIENCE = 3
ENCODER_LR = 2e-5
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
WARMUP_RATIO = 0.1
print('configured encoders:', ENCODERS)

In [ ]:
df = load_dataframe(DATA_PATH)
train_df, val_df, test_df = prepare_splits(df, seed=PRIMARY_SEED)

print(f'rows: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
pd.DataFrame([
    {'split': 'train', 'rows': len(train_df)},
    {'split': 'val', 'rows': len(val_df)},
    {'split': 'test', 'rows': len(test_df)},
]).to_csv(ARTIFACT_DIR / 'transformer_split_metadata.csv', index=False)
display(train_df[['resume_text', 'job_text', 'matched_score']].head(2))

In [ ]:
length_df = summarize_token_lengths(train_df)
print(length_df.describe(percentiles=[0.5, 0.9, 0.95]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(length_df['resume_tokens'], bins=40, ax=axes[0], color='#2a9d8f')
axes[0].axvline(length_df['resume_tokens'].quantile(0.95), color='black', linestyle='--', label='p95')
axes[0].set_title('Resume Token Lengths')
axes[0].legend()
sns.histplot(length_df['job_tokens'], bins=30, ax=axes[1], color='#e76f51')
axes[1].axvline(length_df['job_tokens'].quantile(0.95), color='black', linestyle='--', label='p95')
axes[1].set_title('Job Token Lengths')
axes[1].legend()
plt.tight_layout()
savefig(fig, 'token_length_distributions.png')
plt.show()

In [ ]:
runs = []
for encoder_name in ENCODERS:
    for variant in VARIANTS:
        print(f"\\n=== {encoder_name} | {variant['name']} ===")
        result = run_experiment(
            train_df=train_df,
            val_df=val_df,
            test_df=test_df,
            encoder_name=encoder_name,
            use_structured=variant['use_structured'],
            use_lexical=variant['use_lexical'],
            device=device,
            batch_size=BATCH_SIZE,
            max_length=MAX_LENGTH,
            resume_word_limit=RESUME_WORD_LIMIT,
            job_word_limit=JOB_WORD_LIMIT,
            epochs=EPOCHS,
            encoder_lr=ENCODER_LR,
            head_lr=HEAD_LR,
            weight_decay=WEIGHT_DECAY,
            patience=PATIENCE,
            warmup_ratio=WARMUP_RATIO,
        )
        best_epoch = int(result['history'].sort_values('val_loss').iloc[0]['epoch'])
        run_record = {
            'encoder': encoder_name,
            'variant': result['variant'],
            'best_epoch': best_epoch,
            'val_r2': float(result['history'].sort_values('val_loss').iloc[0]['val_r2']),
            'val_mse': float(result['history'].sort_values('val_loss').iloc[0]['val_mse']),
            'test_r2': result['test_metrics']['r2'],
            'test_mse': result['test_metrics']['mse'],
            'test_mae': result['test_metrics']['mae'],
            'train_rows': result['split_sizes']['train'],
            'val_rows': result['split_sizes']['val'],
            'test_rows': result['split_sizes']['test'],
            'seed': result['seed'],
        }
        result['summary'] = run_record
        runs.append(result)
        print(run_record)

In [ ]:
results_df = pd.DataFrame([run['summary'] for run in runs]).sort_values(['encoder', 'variant']).reset_index(drop=True)
display(results_df)
results_df.to_csv(ARTIFACT_DIR / 'transformer_results.csv', index=False)
for run in runs:
    safe_name = run['encoder'].split('/')[-1].replace('-', '_') + '_' + run['variant'].replace('-', '_')
    run['history'].to_csv(ARTIFACT_DIR / f'history_{safe_name}.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=results_df, x='encoder', y='test_r2', hue='variant', ax=axes[0])
axes[0].set_title('Test R² by Encoder and Variant')
axes[0].tick_params(axis='x', rotation=20)
sns.barplot(data=results_df, x='encoder', y='test_mse', hue='variant', ax=axes[1])
axes[1].set_title('Test MSE by Encoder and Variant')
axes[1].tick_params(axis='x', rotation=20)
sns.barplot(data=results_df, x='encoder', y='test_mae', hue='variant', ax=axes[2])
axes[2].set_title('Test MAE by Encoder and Variant')
axes[2].tick_params(axis='x', rotation=20)
for ax in axes[1:]:
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()
plt.tight_layout()
savefig(fig, 'transformer_encoder_variant_comparison.png')
plt.show()

In [ ]:
baseline_df = pd.DataFrame([
    {'model': 'Structured only', 'family': 'historical', 'test_r2': 0.0939, 'test_mse': 0.0251, 'test_mae': np.nan},
    {'model': 'Separate TF-IDF', 'family': 'historical', 'test_r2': 0.4792, 'test_mse': 0.0144, 'test_mae': np.nan},
    {'model': 'Combined TF-IDF', 'family': 'historical', 'test_r2': 0.4938, 'test_mse': 0.0140, 'test_mae': np.nan},
    {'model': 'Hybrid TF-IDF + structured', 'family': 'historical', 'test_r2': 0.5256, 'test_mse': 0.0131, 'test_mae': np.nan},
    {'model': 'Plain LSTM (DOLMA-backed)', 'family': 'historical', 'test_r2': 0.2910, 'test_mse': 0.0196, 'test_mae': 0.1110},
    {'model': 'Hybrid BiLSTM (DOLMA-backed)', 'family': 'historical', 'test_r2': 0.6679, 'test_mse': 0.009184, 'test_mae': 0.074484},
])

transformer_compare_df = results_df.copy()
transformer_compare_df['model'] = transformer_compare_df['encoder'].str.replace('sentence-transformers/', '', regex=False) + ' | ' + transformer_compare_df['variant']
transformer_compare_df['family'] = 'transformer'
transformer_compare_df = transformer_compare_df[['model', 'family', 'test_r2', 'test_mse', 'test_mae']]

comparison_df = pd.concat([baseline_df, transformer_compare_df], ignore_index=True)
comparison_df = comparison_df.sort_values('test_r2', ascending=False).reset_index(drop=True)
display(comparison_df)
comparison_df.to_csv(ARTIFACT_DIR / 'historical_model_comparison.csv', index=False)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=comparison_df, x='model', y='test_r2', hue='family', ax=ax)
ax.set_title('Test R² Across Historical and Transformer Models')
ax.set_xlabel('')
ax.set_ylabel('Test R²')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
savefig(fig, 'historical_vs_transformer_r2.png')
plt.show()

In [ ]:
for run in runs:
    history_df = run['history']
    analysis_df = run['analysis_df']
    short_name = run['encoder'].split('/')[-1]
    title = f"{short_name} | {run['variant']}"
    safe_name = f"{short_name}_{run['variant']}".replace('-', '_')

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(history_df['epoch'], history_df['train_loss'], marker='o', label='train')
    axes[0].plot(history_df['epoch'], history_df['val_loss'], marker='o', label='val')
    axes[0].set_title(f'{title} Loss Curve')
    axes[0].legend()

    sns.scatterplot(data=analysis_df, x='matched_score', y='predicted_score', alpha=0.35, s=20, ax=axes[1])
    min_score = analysis_df['matched_score'].min()
    max_score = analysis_df['matched_score'].max()
    axes[1].plot([min_score, max_score], [min_score, max_score], 'r--')
    axes[1].set_title(f"{title} Actual vs Predicted")

    sns.histplot(analysis_df['residual'], bins=30, kde=True, ax=axes[2], color='#577590')
    axes[2].axvline(0.0, color='black', linestyle='--')
    axes[2].set_title(f'{title} Residuals')
    plt.tight_layout()
    savefig(fig, f'{safe_name}_overview.png')
    plt.show()

    error_bins = analysis_df.assign(score_bin=pd.cut(analysis_df['matched_score'], bins=np.linspace(0, 1, 6), include_lowest=True))
    error_summary = error_bins.groupby('score_bin', observed=False)['abs_error'].mean().reset_index()
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=error_summary, x='score_bin', y='abs_error', color='#264653', ax=ax)
    ax.set_title(f'{title} Mean Absolute Error by True-Score Bin')
    ax.tick_params(axis='x', rotation=20)
    plt.tight_layout()
    savefig(fig, f'{safe_name}_error_by_bin.png')
    plt.show()

    analysis_df.to_csv(ARTIFACT_DIR / f'analysis_{safe_name}.csv', index=False)
    torch.save(run['model'].state_dict(), ARTIFACT_DIR / f'{safe_name}.pt')

In [ ]:
combined_analysis = pd.concat([run['analysis_df'] for run in runs], ignore_index=True)
best_examples = combined_analysis.sort_values('abs_error').groupby(['encoder', 'variant'], group_keys=False).head(3)
worst_examples = combined_analysis.sort_values('abs_error', ascending=False).groupby(['encoder', 'variant'], group_keys=False).head(3)

print('best predicted examples')
display(best_examples[['encoder', 'variant', 'matched_score', 'predicted_score', 'abs_error', 'resume_text', 'job_text']])
print('worst predicted examples')
display(worst_examples[['encoder', 'variant', 'matched_score', 'predicted_score', 'abs_error', 'resume_text', 'job_text']])